# Plant Guard AI — reproducible held-out ONNX evaluation

This notebook evaluates the existing 38-class ONNX model on the fixed PlantVillage test manifest only. It does not train or change model weights. Keep `plantvillage_test.csv`, `classes.json`, `efficientnet_v2_s_best.onnx`, and its `.onnx.data` file together under the project paths below. The dataset archive stays in Drive and is read in place; no new split is created.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # Colab will request Google authentication
PROJECT = '/content/drive/MyDrive/PlantGuard-AI'  # Set to the uploaded/synced project folder
MODEL_DIR = f'{PROJECT}/backend/ml/models'
STAGE3 = f'{PROJECT}/backend/ml/experiments/stage3_preflight'
ARCHIVE = f'{PROJECT}/datasets/plantvillage-source.zip'


In [ ]:
%pip -q install onnxruntime-gpu pillow numpy pandas scikit-learn matplotlib
import os, json, csv, zipfile, time
from pathlib import Path
from io import BytesIO
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import onnxruntime as ort
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
print('ONNX Runtime', ort.__version__)
print('Available providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'Select a Colab GPU runtime before evaluating.'


In [ ]:
classes = json.load(open(f'{MODEL_DIR}/classes.json', encoding='utf-8'))['classes']
assert len(classes) == len(set(classes)) == 38
rows = pd.read_csv(f'{STAGE3}/plantvillage_test.csv')
assert len(rows) == 8154, f'Unexpected test rows: {len(rows)}'
assert set(rows['split']) == {'test'}
assert set(rows['class_name']) == set(classes)
assert rows['member'].is_unique
print(rows.groupby('class_name').size().reindex(classes).to_string())
model_path=f'{MODEL_DIR}/efficientnet_v2_s_best.onnx'
data_path=model_path+'.data'
assert os.path.isfile(model_path) and os.path.isfile(data_path)
sess=ort.InferenceSession(model_path, providers=['CUDAExecutionProvider','CPUExecutionProvider'])
print('Active providers:',sess.get_providers())
assert sess.get_inputs()[0].shape[1:] == [3,224,224]
assert sess.get_outputs()[0].shape[-1] == 38


In [ ]:
mean=np.array([0.485,0.456,0.406],dtype=np.float32)
std=np.array([0.229,0.224,0.225],dtype=np.float32)
def prep(im):
    im=ImageOps.exif_transpose(im).convert('RGB').resize((224,224),Image.Resampling.BILINEAR)
    a=np.asarray(im,dtype=np.float32)/255.0
    a=(a-mean)/std
    return np.transpose(a,(2,0,1))
label_to_id={c:i for i,c in enumerate(classes)}
y_true=[]; y_pred=[]; batch=[]; batch_y=[]; batch_size=128
# One persistent ZIP handle avoids reparsing the 163k-member central directory.
with zipfile.ZipFile(ARCHIVE) as z:
    available=set(z.namelist())
    for row in rows.itertuples(index=False):
        if row.member not in available: raise FileNotFoundError(row.member)
        with Image.open(BytesIO(z.read(row.member))) as im: batch.append(prep(im))
        batch_y.append(label_to_id[row.class_name])
        if len(batch)==batch_size:
            logits=sess.run(None,{sess.get_inputs()[0].name:np.stack(batch)})[0]
            y_pred.extend(np.argmax(logits,axis=1).tolist()); y_true.extend(batch_y)
            batch.clear(); batch_y.clear()
    if batch:
        logits=sess.run(None,{sess.get_inputs()[0].name:np.stack(batch)})[0]
        y_pred.extend(np.argmax(logits,axis=1).tolist()); y_true.extend(batch_y)
assert len(y_true)==len(rows)==8154


In [ ]:
names=classes
metrics=classification_report(y_true,y_pred,labels=list(range(38)),target_names=names,output_dict=True,zero_division=0)
cm=confusion_matrix(y_true,y_pred,labels=list(range(38)))
summary={'images':len(y_true),'classes':38,'accuracy':accuracy_score(y_true,y_pred),'macro_precision':metrics['macro avg']['precision'],'macro_recall':metrics['macro avg']['recall'],'macro_f1':metrics['macro avg']['f1-score'],'weighted_f1':metrics['weighted avg']['f1-score'],'providers':sess.get_providers(),'test_manifest':'plantvillage_test.csv'}
print(json.dumps(summary,indent=2))
pd.DataFrame([{'class_name':c,**metrics[c]} for c in names]).to_csv(f'{STAGE3}/colab_onnx_per_class_metrics.csv',index=False)
pd.DataFrame(cm,index=names,columns=names).to_csv(f'{STAGE3}/colab_onnx_confusion_matrix.csv')
json.dump({'summary':summary,'classification_report':metrics},open(f'{STAGE3}/colab_onnx_evaluation.json','w'),indent=2)
fig,ax=plt.subplots(figsize=(18,16)); im=ax.imshow(cm,cmap='Blues'); ax.set_xticks(range(38),names,rotation=90,fontsize=6); ax.set_yticks(range(38),names,fontsize=6); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); fig.colorbar(im); fig.tight_layout(); fig.savefig(f'{STAGE3}/colab_onnx_confusion_matrix.png',dpi=180); plt.show()
from google.colab import files
for p in ['colab_onnx_evaluation.json','colab_onnx_per_class_metrics.csv','colab_onnx_confusion_matrix.csv','colab_onnx_confusion_matrix.png']:
    files.download(f'{STAGE3}/{p}')
